# Local models with LM Studio

A local model is a file of numbers plus a program that does math on them, running entirely on your Mac. In this notebook you'll learn what those files are, why quantization decides which models fit in memory, and how to judge a model's speed in the LM Studio app. Then you'll start LM Studio's local server and call it from Python with the same `openai` client you'd use against OpenAI.

**In this notebook**

1. What you're actually downloading
2. Quantization, and why it decides what fits
3. Hands-on in the LM Studio app
4. Start the local server
5. Connect
6. Your first request
7. The model has no memory
8. Streaming and speed
9. Temperature, measured
10. A helper you'll reuse

Sections 1 and 2 are reading plus one small calculation. Section 3 happens in the LM Studio app. Sections 4 to 10 are code against the local server.

**Before you start**

- Module 00 is done, including `uv add openai matplotlib`. This notebook imports `openai`.
- LM Studio (the classic app, not Bionic) is installed. Section 3 downloads one or two models of about 4 to 9 GB each. Budget around two hours in total, most of it in the app.
- From section 5 on, LM Studio must be open with a chat (instruct) model loaded and its server running on port 1234. Section 4 shows how and checks it for you.
- The kernel is the `.venv` inside `ML_Lab` (top right of VS Code).

## 1. What you're actually downloading

A model is two things: a file of weights and a runtime. The weights are billions of numbers learned during training, stored as big matrices. The runtime is the program that multiplies your prompt through those matrices to produce output. LM Studio is a runtime plus a GUI. The weight files come from Hugging Face.

Three properties decide whether a model is useful to you.

Parameter count. The "7B" or "8B" in a model's name is how many weights it has: 8B means about 8 billion numbers. More parameters generally means a more capable model, and also more memory and slower output, because every generated token has to touch every weight once.

Base vs. instruct. A base model was only trained to predict the next token on internet text. Ask it a question and it may just continue your text with more questions. An instruct model (also labeled "chat" or "it") was trained further to follow instructions and hold a conversation. For chatting, always pick instruct. Base models matter later, when you fine-tune.

Format. LM Studio shows two kinds of files. GGUF is a single file (weights, tokenizer, and metadata bundled together) that runs on the llama.cpp engine and works on any computer. MLX is Apple's format: a folder with a `config.json`, tokenizer files, and `.safetensors` weight files, run by Apple's MLX library. MLX is usually faster on Apple Silicon, so on your Mac prefer MLX when both exist.

What to notice: none of this is magic. When you download a model you're downloading a few gigabytes of numbers, and the only question for your hardware is whether those numbers, plus some working memory, fit on the GPU. That's section 2.

## 2. Quantization, and why it decides what fits

Each weight is normally stored as a 16-bit floating-point number (F16 or BF16), which takes 2 bytes. Quantization stores each weight in fewer bits. That shrinks the file and speeds up generation, at a small cost in quality.

### The size formula

$$\text{bytes} \approx N \times \frac{b}{8}$$

Here $N$ is the number of parameters, $b$ is the number of bits per weight, and dividing by 8 converts bits to bytes (8 bits in a byte). For an 8B model at 4 bits: $8 \times 10^9 \times \frac{4}{8} = 4 \times 10^9$ bytes, which is 4 GB. A handy shortcut: with $N$ in billions, $N \times b / 8$ comes out directly in GB.

Real files run a little bigger than the formula. A 4-bit weight can only take 16 different values ($2^4 = 16$), so the format stores, for each small group of weights (64 in MLX's default), a scale $s$ and an offset $z$ in 16-bit precision. Each weight is kept as a small integer $q$ between 0 and 15 and rebuilt as

$$w \approx s \cdot q + z$$

That's two 16-bit numbers (32 bits) shared by 64 weights, or an extra $32 / 64 = 0.5$ bits per weight. So "4-bit" really costs about 4.5 bits per weight, and an 8B model at 4-bit is about 4.5 GB, not 4. GGUF's Q4_K_M lands at a similar size.

Rough sizes for a 7B to 8B model:

| Precision | Name you'll see | Approximate size | Quality |
|---|---|---|---|
| 16-bit | F16, BF16 | ~15 GB | Reference |
| 8-bit | Q8_0, 8bit | ~8 GB | Almost identical |
| ~5-bit | Q5_K_M | ~5.5 GB | Very close |
| ~4-bit | Q4_K_M, 4bit | ~4.5 GB | Small drop, usually the best trade |
| 3-bit and below | Q3, Q2 | ~3 GB or less | Noticeably worse |

### The KV cache

The weights aren't the only thing in memory. While generating, the model keeps a key vector and a value vector for every token it has seen, in every layer, so it doesn't recompute them for each new token. You'll build the attention that produces these keys and values in module 06. Their total size is

$$\text{KV bytes} = 2 \times L \times H_{kv} \times d_{head} \times T \times p$$

where the 2 counts keys and values, $L$ is the number of layers, $H_{kv}$ is the number of key/value heads, $d_{head}$ is the size of each head's vector, $T$ is the number of tokens in the context, and $p$ is bytes per number (2 for 16-bit). For Llama 3.1 8B, $L = 32$, $H_{kv} = 8$, and $d_{head} = 128$, so each token costs $2 \times 32 \times 8 \times 128 \times 2 = 131{,}072$ bytes, about 0.13 MB. An 8,192-token context is about 1.1 GB, and a 32,768-token context is about 4.3 GB. The cache grows linearly with conversation length, which is why long chats eat memory.

### The memory rule

$$\text{weights} + \text{KV cache} \le \text{GPU memory budget}$$

On Apple Silicon the CPU and GPU share one pool of RAM, but macOS only lets the GPU use part of it, roughly two-thirds to three-quarters. Your M5 has 24 GB of unified memory and Metal lets the GPU use about 17.8 GB by default (about 74%). That 17.8 GB is your real budget for weights plus KV cache. On another Mac, check the RAM under Apple menu > About This Mac and take about two-thirds to three-quarters of it.

| Your RAM | Comfortable model size |
|---|---|
| 8 GB | 1B to 3B at 4-bit |
| 16 GB | up to 8B at 4-bit |
| 24 to 32 GB | up to 14B at 4-bit, or 8B at 8-bit |
| 48 GB+ | 30B-class at 4-bit |

The next cell runs the formula for common model sizes and checks each one against your budget.

In [ ]:
# Rough memory math for your M5. Pure Python, no model needed.
GPU_BUDGET_GB = 17.8  # what Metal lets the GPU use on a 24 GB M5 by default
KV_CACHE_GB = 2.0     # allowance for context: about 16k tokens on an 8B model like Llama 3.1 8B

# bytes = parameters x bits / 8. With parameters in billions, that comes out directly in GB.
# Quantized formats also store a scale and offset for every group of 64 weights, which adds
# about half a bit per weight. So 8-bit really costs ~8.5 bits and 4-bit ~4.5 bits.
EFFECTIVE_BITS = {16: 16.0, 8: 8.5, 4: 4.5}

def verdict(weights_gb):
    spare = GPU_BUDGET_GB - (weights_gb + KV_CACHE_GB)
    if spare >= 2:
        return "fits"
    if weights_gb < GPU_BUDGET_GB:
        return "tight"  # loads, but only with a short context and nothing else on the GPU
    return "no"

print(f"{'model':>6}" + "".join(f"{f'{bits}-bit':>18}" for bits in EFFECTIVE_BITS))
for params_b in [1, 3, 8, 14, 32]:
    row = f"{params_b:>5}B"
    for bits, eff_bits in EFFECTIVE_BITS.items():
        weights_gb = params_b * eff_bits / 8
        row += f"{weights_gb:>9.1f} GB {verdict(weights_gb):>5}"
    print(row)
print(f"\nGPU budget {GPU_BUDGET_GB} GB has to hold the weights plus {KV_CACHE_GB} GB of KV cache. "
      "'fits' leaves at least 2 GB to spare.")

What to notice: on your 24 GB M5, an 8B model fits comfortably at 4-bit (about 4.5 GB) or 8-bit (about 8.5 GB), and a 14B model fits at 4-bit (about 8 GB). An 8B model in 16-bit is tight: the weights alone take 16 GB, which leaves almost nothing for context. A 14B model at 8-bit (about 15 GB) is tight for the same reason. A 32B model doesn't fit even at 4-bit, so for that size you'd rent a GPU. LM Studio shows a fit estimate next to each download, so you won't calculate this every time, but now you know where that estimate comes from.

## 3. Hands-on in the LM Studio app

This section is all in the LM Studio app. Work through the steps in order and fill in the table in the next cell as you go.

1. Download a first model. Open the search (Discover) view and look for an instruct model in MLX 4-bit. From section 2 you know an 8B model fits easily, so anything from 3B to 8B works. Qwen and Llama instruct models are good starting points, for example Qwen2.5 7B Instruct or Llama 3.1 8B Instruct. Before you download, read the model card: who made it, its context length, and its license. Pick a plain instruct model, not a reasoning model (one that shows a thinking step, like Qwen3): reasoning models return their thinking separately, so through the API the reply text can come back empty and the cells below would print nothing.
2. Chat with it. Load the model and open the chat view. Find the settings panel and locate the system prompt, temperature, and context length.
3. Watch the numbers. After each reply LM Studio shows tokens per second and time to first token. Write them down in the table. Those two numbers are what you'll care about when you serve a model to real users: time to first token is how long the user stares at nothing, and tokens per second is how fast the answer streams in.
4. Temperature experiment. Ask "Write one sentence describing a basketball game." five times at temperature 0, then five times at temperature 1.5. At 0 the model always picks the most likely next token, so the outputs come out the same or nearly the same. At higher temperature it samples from more of the probability distribution, so you get more variety and more nonsense.
5. System prompt experiment. Set the system prompt to "You are a terse senior engineer. Answer in at most two sentences." and ask a question. Then clear it and ask again. The system prompt is the cheapest way to change a model's behavior. Fine-tuning (module 07) is the expensive way.
6. Quantization experiment. Download the same model at a different quantization (for example 8-bit if you used 4-bit). Ask both the same hard question, like a tricky logic puzzle or a piece of code to debug. Compare speed, memory use (watch the Memory tab in Activity Monitor), and answer quality.
7. Find the limits. Paste in something long (a full lecture transcript or a long article) and ask for a summary. Watch time to first token rise and speed drop as the context grows. That's the KV cache from section 2 at work.

**Your numbers.** Double-click this cell to edit it, fill in a row per run, then press Shift+Enter to render it.

| Model and quantization | Prompt (short or long) | Time to first token (s) | Tokens/sec | Memory in Activity Monitor (GB) | Notes on quality |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |

What to notice: expect the 8-bit model to be noticeably slower than the 4-bit one, roughly in proportion to its size (often a bit less, since 4-bit weights cost some extra math to unpack). Generating each token means reading every weight from memory once, so on a Mac, speed is mostly limited by memory bandwidth, and a file twice as big takes about twice as long per token. The long-prompt run should show a much higher time to first token, because the model has to process the whole prompt before it can produce anything.

## 4. Start the local server

This is the most important part for building. LM Studio can serve the loaded model through an HTTP API that matches OpenAI's format exactly, so any code written for OpenAI works against your local model by changing one URL.

1. In the app: open the Developer view (the terminal-style icon in the left sidebar) and switch the server on. It runs at `http://localhost:1234`.
2. Or from a terminal, with the `lms` CLI that ships with LM Studio:

```bash
lms ls                 # models you've downloaded
lms load               # pick a model to load (interactive)
lms ps                 # what's loaded right now
lms server start       # start the API server
lms server status
lms unload --all       # free the memory when you're done
```

If your shell says `lms` isn't found, run `~/.lmstudio/bin/lms bootstrap` once, then open a new terminal tab. `lms load` asks you to pick a model interactively, so run it in a terminal, not in a notebook cell.

The next cell checks whether the server is reachable. It sends a plain HTTP request to `/v1/models`, the route that lists what the server can serve, using only Python's standard library. It doesn't start or load anything.

In [ ]:
import json
import urllib.request

SERVER = "http://localhost:1234/v1"

try:
    with urllib.request.urlopen(f"{SERVER}/models", timeout=3) as resp:
        ids = [m["id"] for m in json.load(resp)["data"]]
    print(f"LM Studio server is up at {SERVER}. Models it reports:")
    for model_id in ids:
        print("   ", model_id)
    if not [i for i in ids if "embed" not in i]:
        print("No chat model listed. Load an instruct model in the app or with `lms load`.")
except (OSError, ValueError, KeyError) as e:
    print(f"Can't reach the LM Studio server at {SERVER} ({e}).")
    print("Open LM Studio and switch the server on in the Developer view, or run `lms server start`.")

In [ ]:
# Optional: the same check through the CLI. If lms isn't installed this just prints an error.
!~/.lmstudio/bin/lms server status
!~/.lmstudio/bin/lms ps

What to notice: the base URL is `http://localhost:1234/v1`. The `/v1` is part of OpenAI's API layout, and every OpenAI-compatible server (LM Studio, Ollama, vLLM, `mlx_lm.server`) uses it. If LM Studio's just-in-time (JIT) model loading setting is on, `/v1/models` lists every downloaded model rather than only the loaded one, and a request that names an unloaded model loads it first. `lms ps` shows what's actually in memory. When you finish a session, `lms unload --all` gives that memory back, which matters before module 07's training.

Don't move on until the check above says the server is up.

## 5. Connect

You'll use the official `openai` Python package. It doesn't care whose server it talks to, only that the server speaks the same API. That's why the same code works against OpenAI, LM Studio, Ollama, vLLM, and most hosted open-model providers. The only changes are `base_url` and the model name.

In [ ]:
from openai import OpenAI

# LM Studio ignores the API key, but the client requires one, so any string works.
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

models = client.models.list()
for m in models.data:
    print(m.id)

That's every model LM Studio can serve. If you see a connection error instead, the server isn't running (go back to section 4).

In [ ]:
# LM Studio ships with an embedding model that shows up in this list but can't chat, so skip it.
chat_models = [m.id for m in models.data if "embed" not in m.id]
MODEL = chat_models[0]  # if this picks the wrong one, paste the exact id of your loaded model here
print("Using:", MODEL)

## 6. Your first request

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In two sentences, what is quantization in machine learning?"},
    ],
    temperature=0.7,
    max_tokens=200,
)

print(response.choices[0].message.content)
print()
print(response.usage)

What came back:

- `choices[0].message.content` is the text.
- `usage` counts tokens. `prompt_tokens` is what you sent and `completion_tokens` is what the model generated. Hosted APIs bill on exactly these two numbers, so get used to watching them.

`messages` is a list of turns, each with a role. `system` sets behavior, `user` is the human, and `assistant` is the model's earlier replies.

## 7. The model has no memory

This is the most misunderstood thing about LLM APIs. The model is stateless. Every request is independent, and the only "memory" is whatever history you send back in `messages`. Chat apps feel like they remember because the app resends the whole conversation every time.

In [ ]:
# Request 1: tell it something
first = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Pujan and I'm building an app called Darvis."}],
    max_tokens=100,
)
reply = first.choices[0].message.content
print("Model:", reply)

# Request 2: a brand new request with no history
forgetful = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What's my name and what am I building?"}],
    max_tokens=100,
)
print("\nWithout history:", forgetful.choices[0].message.content)

# Request 3: same question, but we send the history ourselves
history = [
    {"role": "user", "content": "My name is Pujan and I'm building an app called Darvis."},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "What's my name and what am I building?"},
]
remembers = client.chat.completions.create(model=MODEL, messages=history, max_tokens=100)
print("\nWith history:", remembers.choices[0].message.content)

This is also why long conversations get slower and more expensive: the prompt grows with every turn.

## 8. Streaming and speed

Chat UIs print tokens as they arrive instead of waiting for the full answer. With `stream=True` you get chunks back one at a time. This cell also measures the two numbers that matter when you serve a model to users: time to first token (how long before anything appears) and tokens per second.

In [ ]:
import time

start = time.perf_counter()
first_token_time = None
chunks = 0

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what a neural network is to a high school student in about 100 words."}],
    stream=True,
    max_tokens=250,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        chunks += 1
        print(delta, end="", flush=True)

end = time.perf_counter()
print(f"\n\nTime to first token: {first_token_time - start:.2f}s")
print(f"Approx tokens/sec:   {chunks / (end - first_token_time):.1f}  (one chunk is roughly one token)")

## 9. Temperature, measured

Same prompt, five samples at each temperature. Count how many distinct answers you get.

In [ ]:
prompt = "Give me a name for a coffee shop run by robots. Reply with the name only."

for temp in [0.0, 0.7, 1.5]:
    outputs = []
    for _ in range(5):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=20,
        )
        outputs.append(r.choices[0].message.content.strip())
    print(f"temperature={temp}: {len(set(outputs))} distinct of 5")
    for o in outputs:
        print("   ", o)

Under the hood the model produces a probability for every token in its vocabulary. Temperature divides the raw scores (logits) before they're turned into probabilities. Low temperature sharpens the distribution so the top token almost always wins. High temperature flattens it so unlikely tokens get picked more often. You'll implement exactly this in module 06.

Rules of thumb: use 0 to 0.3 for extraction, classification, and code, and 0.7 to 1.0 for writing and brainstorming.

## 10. A helper you'll reuse

In [ ]:
def chat(prompt, system=None, temperature=0.7, max_tokens=400, model=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model=model or MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content

print(chat("What is 17 * 23? Show your work briefly.", temperature=0))

## Wrap-up

**Why this matters.** Every request to a hosted model costs money per token and sends your users' data to someone else's servers. Once you can serve a model behind an OpenAI-compatible API, swapping a hosted model for your own is a config change in a backend (a base URL and a model name), not a rewrite. Module 08 does exactly that with the model you fine-tune in module 07. Knowing the memory math also tells you, before you spend anything, which models you can run on a laptop and which need rented GPUs, and that is a cost decision every AI product makes.

**You're done when** you can look at a model's parameter count and quantization and say whether it fits in your 17.8 GB GPU budget, start LM Studio's server (from the app or with `lms`), and call it from Python with streaming while reading off time to first token and tokens per second.

**Going further.** The LM Studio docs at lmstudio.ai/docs cover the server, the `lms` CLI, and structured output in more detail. `lms --help` lists every CLI command. You'll implement temperature sampling yourself in module 06.

---
# Exercises

Try each one before scrolling to the solutions.

**Exercise 1: Benchmark.** Write `benchmark(model_id, runs=3)` that sends the same prompt `runs` times without streaming and returns the average generation speed in tokens per second, using `response.usage.completion_tokens` and a timer. Load a 4-bit and an 8-bit version of the same model in LM Studio and compare them.

**Exercise 2: A mini Darvis.** Put this data in a system prompt and ask the model questions about it:

```
CS 3114 | Prof. A | avg GPA 2.9 | A rate 31%
CS 3114 | Prof. B | avg GPA 3.3 | A rate 48%
CS 3214 | Prof. C | avg GPA 2.7 | A rate 22%
```

Ask "Which CS 3114 professor has better outcomes?" Then ask about CS 4104, which isn't in the data. Does it make something up? Change the system prompt so it says it doesn't know instead. This is grounding, the core idea behind retrieval-augmented generation (RAG). Real RAG adds a search step that picks which rows go into the prompt.

**Exercise 3: Structured output.** Your backend usually needs JSON, not prose. Ask the model to extract `{"course": str, "professor": str, "question_type": str}` from the sentence "is professor b easier than prof a for cs 3114". First try asking for JSON in the prompt and parsing it with `json.loads`. Then try LM Studio's `response_format` with a JSON schema, which forces the output to match.

**Exercise 4: Same code, different server.** Point a second client at Ollama (`http://localhost:11434/v1`) and make the same `chat.completions.create` call against a model from `ollama list`. Only the `base_url`, the API key, and the model name should change. Ollama has to be running (the menu bar app, or `ollama serve` in a terminal), and `ollama list` is empty until you pull a model. If you have none yet, run `ollama pull qwen2.5:1.5b` first (about 1 GB).

---
# Solutions

In [ ]:
# Exercise 1
def benchmark(model_id, runs=3, prompt="Write a 150-word paragraph about the history of basketball."):
    speeds = []
    for _ in range(runs):
        start = time.perf_counter()
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=300,
        )
        elapsed = time.perf_counter() - start
        speeds.append(r.usage.completion_tokens / elapsed)
    return sum(speeds) / len(speeds)

# This number includes time to first token, so it reads a bit lower than LM Studio's own stat.
print(f"{MODEL}: {benchmark(MODEL):.1f} tokens/sec")

In [ ]:
# Exercise 2
DATA = '''
CS 3114 | Prof. A | avg GPA 2.9 | A rate 31%
CS 3114 | Prof. B | avg GPA 3.3 | A rate 48%
CS 3214 | Prof. C | avg GPA 2.7 | A rate 22%
'''

system = (
    "You answer questions about Virginia Tech courses using ONLY the data below. "
    "If the data doesn't contain the answer, say you don't have data for that course. "
    "Never guess.\n\nDATA:\n" + DATA
)

print(chat("Which CS 3114 professor has better outcomes?", system=system, temperature=0))
print("---")
print(chat("Which CS 4104 professor should I take?", system=system, temperature=0))

In [ ]:
# Exercise 3, approach 1: ask nicely, then parse
import json

question = "is professor b easier than prof a for cs 3114"

raw = chat(
    f'Extract fields from this question. Reply with JSON only, no other text, using keys '
    f'"course", "professor", "question_type".\n\nQuestion: {question}',
    temperature=0,
)
print("Raw:", raw)

try:
    print("Parsed:", json.loads(raw))
except json.JSONDecodeError:
    # Small models often wrap JSON in ```json fences. That's why approach 2 exists.
    cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    print("Parsed after cleanup:", json.loads(cleaned))

In [ ]:
# Exercise 3, approach 2: force the shape with a JSON schema
schema = {
    "type": "object",
    "properties": {
        "course": {"type": "string"},
        "professor": {"type": "string"},
        "question_type": {"type": "string", "enum": ["comparison", "recommendation", "requirements", "other"]},
    },
    "required": ["course", "professor", "question_type"],
}

r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the fields from this question: {question}"}],
    response_format={"type": "json_schema", "json_schema": {"name": "query", "schema": schema}},
    temperature=0,
)
print(json.loads(r.choices[0].message.content))

In [ ]:
# Exercise 4
from openai import APIConnectionError

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

try:
    # Ollama sends "data": null, not an empty list, when no model has been pulled yet.
    ollama_models = ollama.models.list().data or []
except APIConnectionError:
    print("Can't reach Ollama at localhost:11434. Open the Ollama app (or run `ollama serve`) and rerun this cell.")
else:
    if not ollama_models:
        print("Ollama is running but has no models. In a terminal run `ollama pull qwen2.5:1.5b`, then rerun this cell.")
    else:
        print([m.id for m in ollama_models])
        OLLAMA_MODEL = ollama_models[0].id
        r = ollama.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": "Say hello in five words."}],
        )
        print(r.choices[0].message.content)